# UMUD — MT NaN Diagnosis (Phase 3)

**GPU notebook** — runs test inference with the same geometry as submission (including **region→line fallback**), then writes per-image diagnostics:

- `mt_diagnosis.csv` — full per-test breakdown
- `mt_diagnosis_summary.json` — aggregate failure counts

Use this to verify A1 before/after submission fixes.


## Configuration


In [ ]:
from pathlib import Path

IMG_SIZE = 256
APO_REGION_THRESHOLD = 0.50

FASC_MODEL_PATH = Path(
    "/kaggle/input/notebooks/ucheozoemena/umud-train-mounted-phase-3/fasc_baseline.pkl"
)
APO_MODEL_PATH = Path(
    "/kaggle/input/notebooks/ucheozoemena/umud-train-apo-mounted-phase-3/apo_baseline.pkl"
)
COMPETITION_DIR = Path(
    "/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data"
)
TEST_DIR = COMPETITION_DIR / "test_images_v2/test_set_v2"


In [ ]:
from pathlib import Path

IMG_SIZE = 256
APO_REGION_THRESHOLD = 0.50

# Pixel → mm scale (Option C). Replace before first leaderboard submit.
MM_PER_PIXEL = 1.0  # placeholder — hunt calibration in Phase 3 work item 5

FASC_MODEL_PATH = Path(
    "/kaggle/input/notebooks/ucheozoemena/umud-train-mounted-phase-3/fasc_baseline.pkl"
)
APO_MODEL_PATH = Path(
    "/kaggle/input/notebooks/ucheozoemena/umud-train-apo-mounted-phase-3/apo_baseline.pkl"
)

COMPETITION_DIR = Path(
    "/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data"
)
TEST_DIR = COMPETITION_DIR / "test_images_v2/test_set_v2"
SAMPLE_SUBMISSION = COMPETITION_DIR / "sample_submission.csv"


In [ ]:
import json

assert FASC_MODEL_PATH.exists(), f"Missing fasc model: {FASC_MODEL_PATH}"
assert APO_MODEL_PATH.exists(), f"Missing apo model: {APO_MODEL_PATH}"
assert TEST_DIR.exists(), f"Missing test dir: {TEST_DIR}"

fasc_learn = load_learner(FASC_MODEL_PATH)
apo_learn = load_learner(APO_MODEL_PATH)
test_paths = list_test_images(TEST_DIR)
print(f"Test images: {len(test_paths)}")


In [ ]:
rows = []
for path in tqdm(test_paths, desc="diagnose test"):
    img_native = load_gray(path)
    h, w = img_native.shape
    pil = open_rgb_256(img_native)
    _, fasc_t, _ = fasc_learn.predict(pil)
    _, apo_t, _ = apo_learn.predict(pil)
    fasc_native = resize_mask_to(tensor_to_mask(fasc_t), h, w)
    apo_native = resize_mask_to(tensor_to_mask(apo_t), h, w)
    apo_style = tag_apo_style(float(apo_native.mean()))
    geo = derive_geometry(fasc_native, apo_native, apo_style)
    rows.append({
        "image_id": path.name,
        "img_h": h,
        "img_w": w,
        "res": f"{h}x{w}",
        "apo_style": apo_style,
        "apo_cov": geo["apo_cov"],
        "apo_fg_pixels": geo["apo_fg_pixels"],
        "fasc_cov": geo["fasc_cov"],
        "fasc_fg_pixels": geo["fasc_fg_pixels"],
        "n_contours": geo["n_contours"],
        "geometry_path": geo["geometry_path"],
        "mt_fail_reason": geo["mt_fail_reason"],
        "mt_fail_reason_primary": geo.get("mt_fail_reason_primary"),
        "mt_px": geo["mt_px"],
        "mt_ok": not np.isnan(geo["mt_px"]),
        "pa_deg": geo["pa_deg"],
        "fl_px": geo["fl_px"],
    })

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/mt_diagnosis.csv", index=False)

summary = {
    "n_test": len(df),
    "mt_nan_rate": float((~df.mt_ok).mean()),
    "by_apo_style": df.groupby("apo_style").mt_ok.mean().to_dict(),
    "by_geometry_path": df.groupby("geometry_path").mt_ok.mean().to_dict(),
    "fail_reason_counts": df.loc[~df.mt_ok, "mt_fail_reason"].value_counts().to_dict(),
    "fail_reason_x_style": pd.crosstab(df.apo_style, df.mt_fail_reason).to_dict(),
    "fallback_line_rescues": int((df.geometry_path == "fallback_line").sum()),
}
Path("/kaggle/working/mt_diagnosis_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
display(df.loc[~df.mt_ok].head(20))
